# NHL Draft Analysis

This notebook loads the draft scores produced by formula.ipynb, applies 
franchise consolidation, adds team colors, and exports clean CSVs for 
the Tableau dashboard.

## Setup

In [1]:
import pandas as pd

df = pd.read_csv('../data/outputs/draft_scores.csv')
print(f"Loaded {len(df)} rows")
df.head()

Loaded 2484 rows


,draft_year,round,pick_number,drafted_by,player_name,position,games_played,points,performance_score,hindsight_rank,plus_minus
0,2005,1,1,Pittsburgh,Sidney Crosby,C,1408.0,1746.0,5406.34,1,0
1,2005,1,2,Anaheim,Bobby Ryan,R,866.0,569.0,2169.01,9,-7
2,2005,1,3,Carolina,Jack Johnson,D,1228.0,342.0,2011.18,12,-9
3,2005,1,4,Minnesota,Benoit Pouliot,L,625.0,263.0,1227.27,21,-17
4,2005,1,6,Columbus,Gilbert Brule,C,299.0,95.0,516.55,39,-33


## Team Rankings

Overall team draft rankings from 2005 to 2017, measured by cumulative 
plus/minus across all picks and all drafts.

In [2]:
team_rankings = df.groupby('drafted_by').agg(
    total_plus_minus=('plus_minus', 'sum'),
    total_picks=('plus_minus', 'count'),
    avg_plus_minus_per_pick=('plus_minus', 'mean')
).round(2).reset_index()

team_rankings = team_rankings.sort_values('total_plus_minus', ascending=False).reset_index(drop=True)
team_rankings.index += 1
team_rankings.columns = ['team', 'total_plus_minus', 'total_picks', 'avg_per_pick']

print("NHL Team Draft Rankings 2005-2017")
print(team_rankings.to_string())

NHL Team Draft Rankings 2005-2017
            team  total_plus_minus  total_picks  avg_per_pick
1       San Jose              4077           81         50.33
2    Los Angeles              3940           88         44.77
3        Detroit              3536           87         40.64
4      Tampa Bay              3382           86         39.33
5     Washington              3196           78         40.97
6        Chicago              3186          104         30.63
7        Toronto              3144           87         36.14
8     New Jersey              3127           84         37.23
9      Nashville              3079           85         36.22
10      Columbus              2927           85         34.44
11  Philadelphia              2871           78         36.81
12     Minnesota              2871           73         39.33
13    Pittsburgh              2815           71         39.65
14        Ottawa              2775           78         35.58
15  NY Islanders              2684  

## Consistency Analysis

Team plus/minus broken down by draft year. Reveals whether a team's 
overall ranking is driven by consistent performance or a few standout 
draft classes.

In [3]:
team_by_year = df.groupby(['drafted_by', 'draft_year']).agg(
    plus_minus=('plus_minus', 'sum'),
    total_picks=('plus_minus', 'count'),
    avg_plus_minus_per_pick=('plus_minus', 'mean')
).round(2).reset_index()

team_by_year = team_by_year.rename(columns={'drafted_by': 'team'})

print(f"team_by_year shape: {team_by_year.shape}")
team_by_year.head(20)

team_by_year shape: (391, 5)


,team,draft_year,plus_minus,total_picks,avg_plus_minus_per_pick
0,Anaheim,2005,6,5,1.20
1,Anaheim,2006,89,5,17.80
2,Anaheim,2007,14,6,2.33
3,Anaheim,2008,119,9,13.22
4,Anaheim,2009,74,6,12.33
5,Anaheim,2010,370,8,46.25
6,Anaheim,2011,205,6,34.17
7,Anaheim,2012,249,7,35.57
8,Anaheim,2013,84,5,16.80
9,Anaheim,2014,218,5,43.60


## Round Analysis

Team plus/minus broken down by draft round. Reveals whether certain 
teams excel at finding late round steals or consistently capitalize 
on early round picks.

In [4]:
team_by_round = df.groupby(['drafted_by', 'round']).agg(
    plus_minus=('plus_minus', 'sum'),
    total_picks=('plus_minus', 'count'),
    avg_plus_minus_per_pick=('plus_minus', 'mean')
).round(2).reset_index()

team_by_round = team_by_round.rename(columns={'drafted_by': 'team'})

print(f"team_by_round shape: {team_by_round.shape}")
team_by_round.head(20)
print(team_by_round.to_string())

team_by_round shape: (216, 5)
             team  round  plus_minus  total_picks  avg_plus_minus_per_pick
0         Anaheim      1        -232           15                   -15.47
1         Anaheim      2        -270           16                   -16.88
2         Anaheim      3         -47           12                    -3.92
3         Anaheim      4         270           10                    27.00
4         Anaheim      5         624           12                    52.00
5         Anaheim      6         595            7                    85.00
6         Anaheim      7         710            6                   118.33
7         Arizona      1        -402           18                   -22.33
8         Arizona      2        -294           13                   -22.62
9         Arizona      3        -108           14                    -7.71
10        Arizona      4         231            9                    25.67
11        Arizona      5         606           11                    5

The round breakdown reveals a consistent pattern across all teams — rounds 
1 through 3 are almost universally negative, while rounds 4 through 7 
produce increasingly large positive scores. This confirms the late round 
asymmetry identified in formula.ipynb: the metric rewards late round steals 
more than it penalizes early round busts, and the cumulative team rankings 
are heavily influenced by late round performance.

To supplement the main ranking, we produce a separate dataset restricted 
to rounds 1 through 3 — the picks where scouting skill is most scrutinized 
and where the late round asymmetry does not apply.

### Early Rounds Ranking — Rounds 1-3 Only

In [5]:
df_early = df[df['round'] <= 3]

team_rankings_early = df_early.groupby('drafted_by').agg(
    total_plus_minus=('plus_minus', 'sum'),
    total_picks=('plus_minus', 'count'),
    avg_plus_minus_per_pick=('plus_minus', 'mean')
).round(2).reset_index()

team_rankings_early = team_rankings_early.sort_values('total_plus_minus', ascending=False).reset_index(drop=True)
team_rankings_early.index += 1
team_rankings_early = team_rankings_early.rename(columns={'drafted_by': 'team'})

print("NHL Team Draft Rankings 2005-2017 — Rounds 1-3 Only")
print(team_rankings_early.to_string())

NHL Team Draft Rankings 2005-2017 — Rounds 1-3 Only
            team  total_plus_minus  total_picks  avg_plus_minus_per_pick
1          Vegas               -17            6                    -2.83
2    Los Angeles              -129           35                    -3.69
3        Detroit              -133           37                    -3.59
4      Nashville              -135           36                    -3.75
5     New Jersey              -170           37                    -4.59
6     Pittsburgh              -205           28                    -7.32
7   Philadelphia              -251           33                    -7.61
8     Washington              -279           29                    -9.62
9       Carolina              -299           38                    -7.87
10     Minnesota              -331           27                   -12.26
11    NY Rangers              -368           33                   -11.15
12     Tampa Bay              -373           38                    -9.82

## Tableau Preparation

All outputs are saved to data/outputs/ for use in the Tableau dashboard.

In [6]:
# Export all CSVs
team_rankings.to_csv('../data/outputs/team_rankings_initial.csv', index=False)
team_by_year.to_csv('../data/outputs/team_by_year_initial.csv', index=False)
team_by_round.to_csv('../data/outputs/team_by_round_initial.csv', index=False)
team_rankings_early.to_csv('../data/outputs/team_rankings_early_initial.csv', index=False)
df.to_csv('../data/outputs/draft_scores_pre_fix.csv', index=False)

print("All files exported successfully!")
print(f"  team_rankings_initial.csv      — {len(team_rankings)} rows")
print(f"  team_by_year_initial.csv       — {len(team_by_year)} rows")
print(f"  team_by_round_initial.csv      — {len(team_by_round)} rows")
print(f"  team_rankings_early_initial.csv — {len(team_rankings_early)} rows")
print(f"  draft_scores_pre_fix.csv         — {len(df)} rows")

All files exported successfully!
  team_rankings_initial.csv      — 31 rows
  team_by_year_initial.csv       — 391 rows
  team_by_round_initial.csv      — 216 rows
  team_rankings_early_initial.csv — 31 rows
  draft_scores_pre_fix.csv         — 2484 rows


## End of Analysis

In [7]:
print("Analysis complete.")

Analysis complete.


All datasets have been prepared and exported to data/outputs/. 
These files serve as the data source for the Tableau dashboard.